In [ ]:
import pickle
from datasets import Dataset

with open('../../raw_data/nmrshiftdb2_2024/mol_nmrshift_nmrshiftdb2_2024_train.pkl', 'rb') as f:
    nmrshiftdb2_train = pickle.load(f)
with open('../../raw_data/nmrshiftdb2_2024/mol_nmrshift_nmrshiftdb2_2024_test.pkl', 'rb') as f:
    nmrshiftdb2_test = pickle.load(f)

key_list = list(nmrshiftdb2_train.keys())
nmrshiftdb2_train_list = [{key: nmrshiftdb2_train[key][i] for key in key_list} for i in range(len(nmrshiftdb2_train[key_list[0]]))]
nmrshiftdb2_test_list = [{key: nmrshiftdb2_test[key][i] for key in key_list} for i in range(len(nmrshiftdb2_test[key_list[0]]))]

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from tqdm import tqdm

def get_element_symbols(mol):
    element_symbols = []  
    for atom in mol.GetAtoms():
        element_symbol = atom.GetSymbol() 
        element_symbols.append(element_symbol)
    return element_symbols

def process_single_entry(entry, seed=42):
    try:
        # 1. Parse SMILES and add hydrogens
        mol = entry['mol']
        smiles = entry['smiles']
        db_id = entry['db_id']
        params = AllChem.ETKDGv3()
        params.randomSeed = seed
        # 2. Embed (use ETKDGv3 if no custom params supplied)
        status = AllChem.EmbedMolecule(mol, params = params)
        if status == -1:
            raise ValueError("Comformer generation failed")
        # 3. MMFF94 geometry optimisation to refine coordinates
        try:
            # some conformer can not use MMFF optimize
            AllChem.MMFFOptimizeMolecule(mol)
            coordinates = mol.GetConformer().GetPositions().astype(np.float32)
        except:
            coordinates = mol.GetConformer().GetPositions().astype(np.float32)
        # 4. Extract atomic symbols
        atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
        # 5. Build placeholder property arrays (edit as needed)
        atoms_target = entry['atom_target'][1:len(atoms)+1]
        atoms_target_mask = []
        for mask in entry['atom_mask'][1:len(atoms)+1]:
            if mask:
                atoms_target_mask.append(1)
            else:
                atoms_target_mask.append(0)

        try:
            inchikey = Chem.MolToInchiKey(mol)
        except:
            inchikey = None

        return {
            'atoms': atoms,
            'coordinates': coordinates,
            'atoms_target': atoms_target,
            'atoms_target_mask': atoms_target_mask,
            'smiles_with_hydrogens': smiles,
            'db_id': db_id,
            'inchikey': inchikey,
        }
    except Exception as e:
        print(f"Error processing entry {entry.get('db_id', 'unknown')}: {e}")
        return None
processed_train_data = [process_single_entry(entry, seed=42) for entry in nmrshiftdb2_train_list]
processed_test_data = [process_single_entry(entry, seed=42) for entry in nmrshiftdb2_test_list]
processed_train_data = [entry for entry in tqdm(processed_train_data) if entry is not None]
processed_test_data = [entry for entry in tqdm(processed_test_data) if entry is not None]

In [ ]:
train_dataset = Dataset.from_list(processed_train_data)
valid_dataset = Dataset.from_list(processed_test_data)
test_dataset = Dataset.from_list(processed_test_data)

from datasets import DatasetDict
nmrshiftdb_dataset = DatasetDict({
    'train': train_dataset,
    'validation': valid_dataset,
    'test': test_dataset,
})
nmrshiftdb_dataset.save_to_disk('../../data/nmrshiftdb2_2025')